## 原始信号，时域&频域

### 导入模型

In [ ]:
import argparse
from model.TSPN import Transparent_Signal_Processing_Network
from trainer.trainer_basic import Basic_plmodel

import torch
from pytorch_lightning import seed_everything
from configs.config import parse_arguments,config_network
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
parser = argparse.ArgumentParser(description='TSPN')
parser.add_argument('--config_dir', type=str, default='configs/a_027_PU/config_basic.yaml',help='The directory of the configuration file')
# 适用于jupyter
meta_args = parser.parse_known_args()[0]
config_dir = meta_args.config_dir
configs,args,path,name = parse_arguments(config_dir, 0)
signal_processing_modules, feature_extractor_modules = config_network(configs,args)
MODEL_DICT = {'TSPN': lambda args: Transparent_Signal_Processing_Network(signal_processing_modules, feature_extractor_modules,args)}
model_plain = MODEL_DICT[args.model](args)
model = Basic_plmodel(model_plain, args)
state_dict = torch.load("./save/test/model_tspn_pu_01.ckpt")
model.load_state_dict(state_dict['state_dict'])
print(model)

### 绘图和计算频率函数

In [ ]:
fault_type_list = ['Norm','Inner Fault','Outer Fault']
def get_TSPN_signal(model, signal,i):
    with torch.no_grad():
        for idx, layer in enumerate(model.signal_processing_layers):
            
            signal = layer(signal)
            if idx == i:
                break
    return signal

def compute_frequency_domain(signal):
    fft_values = torch.fft.rfft(signal)
    fft_values = fft_values.detach().cpu().numpy()
    # 去除直流项
    fft_values[0] = 0
    return np.abs(fft_values)

def plot_signal(signal, savename):
    plt.rcParams['font.family'] = 'Times New Roman'
    
    fig, axs = plt.subplots(4, 2, figsize=(16, 4))
   
    for i in range(4):
        signal_time = F.normalize(signal[i,:],dim=0)
        t = np.linspace(0, 1, signal_time.shape[0])
        signal_time = signal_time.detach().cpu().numpy()
        signal_fft = compute_frequency_domain(signal[i,:])
        axs[i,0].plot(t, signal_time,linewidth=1.0 ,c='peru')
        axs[i,1].plot(signal_fft,linewidth=1.0 ,c='darkolivegreen')
        if i == 3:
            axs[i,0].set_xlabel('Time (s)')
            axs[i,1].set_xlabel('Frequency (Hz)')
        else:
            axs[i,0].set_xticks([])
            axs[i,1].set_xticks([])
    fig.subplots_adjust(wspace=0.1, hspace=0.1)
    # fig.tight_layout()
    plt.savefig(savename)

def plot_ori_signal(number, signal):
    plt.rcParams['font.family'] = 'Times New Roman'
    test_signal = signal[number,:,:].unsqueeze(0)
    
    fig, axs = plt.subplots(12, 2, figsize=(16, 6))
    for i in range(12):
        signal_time = F.normalize(test_signal[0,:,i],dim=0)
        t = np.linspace(0, 1, signal_time.shape[0])
        signal_time = signal_time.detach().cpu().numpy()
        signal_fft = compute_frequency_domain(test_signal[0,:,i])
        axs[i,0].plot(t, signal_time,linewidth=0.5 ,c='peru')
        axs[i,1].plot(signal_fft,linewidth=0.5 ,c='darkolivegreen')
        if i == 2:
            axs[i,0].set_xlabel('Time (s)')
            axs[i,1].set_xlabel('Frequency (Hz)')
    # signal_time = F.normalize(test_signal[0,:,0],dim=0)
    # signal_time = signal_time.detach().cpu().numpy()
    # signal_fft = compute_frequency_domain(test_signal[0,:,0])
    # t = np.linspace(0, 1, 4096)
    # axs[0].plot(t, signal_time,linewidth=0.5 ,c='peru')
    # axs[0].set_xlabel('Time (s)')
    # axs[0].set_ylabel('Amplitude')
    # axs[0].set_title(fault_type_list[number//1222])
    # axs[1].plot(signal_fft,linewidth=0.5 ,c='darkolivegreen')
    # axs[1].set_xlabel('Frequency (Hz)')
    # axs[1].set_ylabel('Amplitude')
    # axs[1].set_title(fault_type_list[number//1222])
    fig.tight_layout()
    plt.savefig('save/figure/PU/tspn/'+fault_type_list[number//1222]+'_original.svg')

### 原始信号和增强信号

In [ ]:
test_data = np.load("C:/Users/CCSLab/Desktop/PU_bearing/PU_N09_M07_F10_data.npy")
test_signal = torch.from_numpy(test_data).cuda().float()
signal_num_list = [1500]
for signal_num in signal_num_list:
    torch.cuda.empty_cache()
    plot_ori_signal(signal_num, test_signal)
    for i in range(1):
        output_signal = get_TSPN_signal(model.network.cuda(), test_signal, i)[signal_num,:,:]
        output_signal = output_signal.T
        plot_signal(output_signal, 'save/figure/pu/tspn/signal' + str(i) + '.svg')
        print(output_signal.shape)
    

In [ ]:
from einops import rearrange
def get_signal_operator_output(signal):
    spl = model.network.signal_processing_layers[0]
    with torch.no_grad():
        x = rearrange(signal, 'b l c -> b c l')
        normed_x = spl.norm(x)
        normed_x = rearrange(normed_x, 'b c l -> b l c')
        # 通过线性层
        
        spl.weight_connection.weight.data = F.softmax((1.0 / spl.temperature) *
                                                       spl.weight_connection.weight.data, dim=0)
        x = spl.weight_connection(normed_x)
        x2 = x
        # 按模块数拆分
        splits = torch.split(x, x.size(2) // spl.module_num, dim=2)

        # 通过模块计算
        outputs = []
        for module, split in zip(spl.signal_processing_modules.values(), splits):
            outputs.append(module(split))
        x = torch.cat(outputs, dim=2)
    return normed_x, x, x2

normx, output_1, weightx = get_signal_operator_output(test_signal)
plot_ori_signal(signal_num, normx)
plot_ori_signal(signal_num, weightx)
plot_ori_signal(signal_num, output_1)